In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make src importable (assumes this notebook lives in notebooks/)
sys.path.insert(0, str(Path().resolve().parent))

from src.pipeline.llm_client import LLMClient
from src.pipeline.orchestrator import ReasoningPipeline

print("Environment, LLMClient, and ReasoningPipeline imports ready.")

Environment, LLMClient, and ReasoningPipeline imports ready.


In [2]:
# Configure the Ollama-backed LLMClient and 5-hop ReasoningPipeline

# If you want to override these, set OLLAMA_MODEL / OLLAMA_HOST in your .env
llm = LLMClient(
    max_tokens=128,
    model="gemma3:1b",
)

pipeline = ReasoningPipeline(llm_client=llm)

print("LLMClient (Ollama) and ReasoningPipeline initialised:")
print(pipeline.get_usage_stats())

LLMClient (Ollama) and ReasoningPipeline initialised:
{'total_calls': 0, 'total_prompt_tokens': 0, 'total_completion_tokens': 0, 'total_tokens': 0}


In [3]:
# Load sentiment_predictions_allday_articles.csv and select ~40 samples

root = Path().resolve().parent
csv_path = root / "sentiment_predictions_allday_articles.csv"
print("Loading:", csv_path)

df = pd.read_csv(csv_path)
print("Total rows:", len(df))
print(df.head(3))

# Sample 40 rows for quick testing
sample_df = df.sample(n=40, random_state=42).reset_index(drop=True)
print("Sampled rows:", len(sample_df))
sample_df.head()

Loading: /Users/longnguyen/Desktop/uoa-group1-c6/sentiment_predictions_allday_articles.csv
Total rows: 293
  published_at  ticker                                              title  \
0   2023-01-12  EURCHF  Euro to benefit from the ECBs pronounced hawki...   
1   2023-01-13  EURCHF  EURCHF to head higher towards 10130 and projec...   
2   2023-01-14  EURCHF  EURCHF reaches 38.2% of the 2 year range as ru...   

   gpt_sentiment_p5  gpt_completion_tokens_p5  gpt_prompt_tokens_p5  \
0                 1                       1.0                 133.0   
1                 1                       1.0                 128.0   
2                 1                       1.0                  82.0   

   gpt_time_p5  gpt_sentiment_p5n  gpt_completion_tokens_p5n  \
0     0.578916                0.5                        3.0   
1     0.909433                0.8                        3.0   
2     0.715290                0.5                       50.0   

   gpt_prompt_tokens_p5n  gpt_time_p5n  gp

,published_at,ticker,title,gpt_sentiment_p5,gpt_completion_tokens_p5,gpt_prompt_tokens_p5,gpt_time_p5,gpt_sentiment_p5n,gpt_completion_tokens_p5n,gpt_prompt_tokens_p5n,gpt_time_p5n,gpt_sentiment_p6n,gpt_completion_tokens_p6n,gpt_prompt_tokens_p6n,gpt_time_p6n,gpt_sentiment_p6,gpt_completion_tokens_p6,gpt_prompt_tokens_p6,gpt_time_p6
0,2023-03-05,EURUSD,EURUSD Price Analysis Bulls eye a test of bear...,1,1.0,101.0,0.611927,0.5,5.0,106.0,1.127559,0.8,31.333333,60.00,6.391485,1.0,8.666667,57.666667,1.394779
1,2023-04-26,EURUSD,EURUSD faces resistance around 10980 after a l...,1,1.0,216.0,0.879438,0.2,3.0,215.0,1.004676,0.6,9.250000,174.00,1.024565,1.0,7.250000,172.250000,1.024152
2,2023-02-22,GBPUSD,GBPUSD Price Analysis Bulls eye another battle...,0,1.0,292.0,0.708211,-0.5,4.0,291.0,1.330337,-0.1,9.250000,236.75,1.158583,0.0,7.250000,235.000000,0.947981
3,2023-03-29,EURUSD,EURUSD Price Analysis Bulls run into key resis...,0,1.0,416.0,0.490239,0.2,3.0,415.0,0.926480,0.1,9.250000,262.25,1.101079,0.0,7.250000,260.500000,0.931855
4,2023-03-21,EURCHF,EURCHF Price Analysis Eyes 10000 as Credit Sui...,1,1.0,106.0,0.626280,0.5,3.0,104.0,0.922714,-0.2,9.400000,189.60,0.994240,-1.0,7.400000,188.200000,0.874306


In [ ]:
# Run the 5-hop ReasoningPipeline on each sampled headline

results = []
for _, row in sample_df.iterrows():
    text = row["title"]
    ticker = row["ticker"]
    try:
        ctx = pipeline.run(text=text, ticker=ticker)
        final = pipeline.get_final_result(ctx)
    except Exception as e:
        final = {"error": str(e)}
    results.append(final)

# Flatten a few key fields into columns for easier analysis
sample_df["hop_sentiment"] = [r.get("sentiment") for r in results]

# Derive a numeric score from the textual hop_sentiment
# negative -> -1, neutral -> 0, positive -> 1

# sample_df["hop_market_implication"] = [r.get("market_implication") for r in results]

sample_df.head()

,published_at,ticker,title,gpt_sentiment_p5,gpt_completion_tokens_p5,gpt_prompt_tokens_p5,gpt_time_p5,gpt_sentiment_p5n,gpt_completion_tokens_p5n,gpt_prompt_tokens_p5n,...,gpt_completion_tokens_p6n,gpt_prompt_tokens_p6n,gpt_time_p6n,gpt_sentiment_p6,gpt_completion_tokens_p6,gpt_prompt_tokens_p6,gpt_time_p6,hop_sentiment,hop_sentiment_score,hop_market_implication
0,2023-03-05,EURUSD,EURUSD Price Analysis Bulls eye a test of bear...,1,1.0,101.0,0.611927,0.5,5.0,106.0,...,31.333333,60.00,6.391485,1.0,8.666667,57.666667,1.394779,Neutral,NaN,Bullish
1,2023-04-26,EURUSD,EURUSD faces resistance around 10980 after a l...,1,1.0,216.0,0.879438,0.2,3.0,215.0,...,9.250000,174.00,1.024565,1.0,7.250000,172.250000,1.024152,Positive,0.6,Bullish
2,2023-02-22,GBPUSD,GBPUSD Price Analysis Bulls eye another battle...,0,1.0,292.0,0.708211,-0.5,4.0,291.0,...,9.250000,236.75,1.158583,0.0,7.250000,235.000000,0.947981,Positive,0.6,Bullish
3,2023-03-29,EURUSD,EURUSD Price Analysis Bulls run into key resis...,0,1.0,416.0,0.490239,0.2,3.0,415.0,...,9.250000,262.25,1.101079,0.0,7.250000,260.500000,0.931855,Positive,NaN,Bullish
4,2023-03-21,EURCHF,EURCHF Price Analysis Eyes 10000 as Credit Sui...,1,1.0,106.0,0.626280,0.5,3.0,104.0,...,9.400000,189.60,0.994240,-1.0,7.400000,188.200000,0.874306,Neutral,NaN,Bullish


In [9]:
def _map_sentiment_to_score(s):
    if s is None:
        return 0
    t = str(s).strip().lower()
    if t.startswith("neg"):
        return -1
    if t.startswith("pos"):
        return 1
    if t.startswith("neu"):
        return 0
    # Fallback: treat anything else as neutral
    return 0


sample_df["hop_sentiment_score"] = sample_df["hop_sentiment"].apply(
    _map_sentiment_to_score
)

sample_df.head()

,published_at,ticker,title,gpt_sentiment_p5,gpt_completion_tokens_p5,gpt_prompt_tokens_p5,gpt_time_p5,gpt_sentiment_p5n,gpt_completion_tokens_p5n,gpt_prompt_tokens_p5n,...,gpt_completion_tokens_p6n,gpt_prompt_tokens_p6n,gpt_time_p6n,gpt_sentiment_p6,gpt_completion_tokens_p6,gpt_prompt_tokens_p6,gpt_time_p6,hop_sentiment,hop_sentiment_score,hop_market_implication
0,2023-03-05,EURUSD,EURUSD Price Analysis Bulls eye a test of bear...,1,1.0,101.0,0.611927,0.5,5.0,106.0,...,31.333333,60.00,6.391485,1.0,8.666667,57.666667,1.394779,Neutral,0,Bullish
1,2023-04-26,EURUSD,EURUSD faces resistance around 10980 after a l...,1,1.0,216.0,0.879438,0.2,3.0,215.0,...,9.250000,174.00,1.024565,1.0,7.250000,172.250000,1.024152,Positive,1,Bullish
2,2023-02-22,GBPUSD,GBPUSD Price Analysis Bulls eye another battle...,0,1.0,292.0,0.708211,-0.5,4.0,291.0,...,9.250000,236.75,1.158583,0.0,7.250000,235.000000,0.947981,Positive,1,Bullish
3,2023-03-29,EURUSD,EURUSD Price Analysis Bulls run into key resis...,0,1.0,416.0,0.490239,0.2,3.0,415.0,...,9.250000,262.25,1.101079,0.0,7.250000,260.500000,0.931855,Positive,1,Bullish
4,2023-03-21,EURCHF,EURCHF Price Analysis Eyes 10000 as Credit Sui...,1,1.0,106.0,0.626280,0.5,3.0,104.0,...,9.400000,189.60,0.994240,-1.0,7.400000,188.200000,0.874306,Neutral,0,Bullish


In [10]:
# Evaluate 5-hop pipeline sentiment vs original CSV sentiment

from src.evaluation.metrics import compute_classification_metrics

# Choose which original sentiment column to compare against
REF_COL = "gpt_sentiment_p6"  # change if you want to compare to another column

print("Using reference column:", REF_COL)

# Map numeric sentiments (-1, 0, 1) to the unified label space expected by metrics
# Our reference columns are already in {-1, 0, 1}, but may contain NaNs
y_true = sample_df[REF_COL]

# For the 5-hop pipeline, we rely on hop_sentiment_score (numeric) if available;
# otherwise you could plug in your own numeric mapping from hop_sentiment text.
y_pred = sample_df["hop_sentiment_score"]

metrics = compute_classification_metrics(y_true, y_pred)
print("Number of valid pairs:", metrics.get("n"))
print("Accuracy:", metrics.get("accuracy"))
print("F1 (macro):", metrics.get("f1_macro"))
print("Precision (macro):", metrics.get("precision_macro"))
print("Recall (macro):", metrics.get("recall_macro"))
print("\nPer-class metrics:")
for label, vals in metrics.get("per_class", {}).items():
    print(label, vals)

Using reference column: gpt_sentiment_p6
Number of valid pairs: 40
Accuracy: 0.475
F1 (macro): 0.47002553313913764
Precision (macro): 0.4696969696969697
Recall (macro): 0.4863247863247863

Per-class metrics:
Negative {'precision': 0.5454545454545454, 'recall': 0.5, 'f1': 0.5217391304347826}
Neutral {'precision': 0.36363636363636365, 'recall': 0.26666666666666666, 'f1': 0.3076923076923077}
Positive {'precision': 0.5, 'recall': 0.6923076923076923, 'f1': 0.5806451612903226}


In [11]:
# Compare 5-hop hop_sentiment_score against ALL GPT sentiment columns from the original CSV

from src.evaluation.metrics import compute_classification_metrics

# Our pipeline prediction (numeric in {-1, 0, 1})
y_pred = sample_df["hop_sentiment_score"]

# Find all GPT sentiment columns in the original allday file
sent_cols = [c for c in sample_df.columns if c.startswith("gpt_sentiment_")]
# Use only discrete sentiment columns (ignore *_n continuous ones)
discrete_sent_cols = [c for c in sent_cols if not c.endswith("n")]

print("Comparing hop_sentiment_score against these reference columns:")
for col in discrete_sent_cols:
    print(" -", col)

results = {}
for col in discrete_sent_cols:
    y_true = sample_df[col]
    metrics = compute_classification_metrics(y_true, y_pred)
    results[col] = metrics

    print("\n=== Reference:", col, "===")
    print("Number of valid pairs:", metrics.get("n"))
    print("Accuracy:", metrics.get("accuracy"))
    print("F1 (macro):", metrics.get("f1_macro"))
    print("Precision (macro):", metrics.get("precision_macro"))
    print("Recall (macro):", metrics.get("recall_macro"))
    print("Per-class metrics:")
    for label, vals in metrics.get("per_class", {}).items():
        print("  ", label, vals)

# 'results' now holds the full metric dict for each GPT sentiment reference column

Comparing hop_sentiment_score against these reference columns:
 - gpt_sentiment_p5
 - gpt_sentiment_p6

=== Reference: gpt_sentiment_p5 ===
Number of valid pairs: 40
Accuracy: 0.425
F1 (macro): 0.3676576006746785
Precision (macro): 0.38552188552188554
Recall (macro): 0.3958333333333333
Per-class metrics:
   Negative {'precision': 0.09090909090909091, 'recall': 0.25, 'f1': 0.13333333333333333}
   Neutral {'precision': 0.45454545454545453, 'recall': 0.25, 'f1': 0.3225806451612903}
   Positive {'precision': 0.6111111111111112, 'recall': 0.6875, 'f1': 0.6470588235294118}

=== Reference: gpt_sentiment_p6 ===
Number of valid pairs: 40
Accuracy: 0.475
F1 (macro): 0.47002553313913764
Precision (macro): 0.4696969696969697
Recall (macro): 0.4863247863247863
Per-class metrics:
   Negative {'precision': 0.5454545454545454, 'recall': 0.5, 'f1': 0.5217391304347826}
   Neutral {'precision': 0.36363636363636365, 'recall': 0.26666666666666666, 'f1': 0.3076923076923077}
   Positive {'precision': 0.5, 'r